# **Libraries import**

In [ ]:
import pandas as pd
import numpy as np
import datetime
import json
import pickle

pd.set_option('display.max_columns', None)

# **Datasets loading**

In [ ]:
calls = pd.read_csv('calls_raw.csv', dtype={'Id':str, 'CONTACTID':str})
spend = pd.read_csv('spend_raw.csv')
deals = pd.read_csv('deals_raw.csv', dtype={'Id': str, 'Contact Name': str})
contacts = pd.read_csv('contacts_raw.csv', dtype={'Id':str})

# **Datasets processing**

## **Contacts**

In [ ]:
contacts.head()

###Datatype changes

In [ ]:
contacts["Created Time"] = pd.to_datetime(contacts["Created Time"], errors="raise")
contacts["Modified Time"] = pd.to_datetime(contacts["Modified Time"], errors="raise")

###Drop duplicates

In [ ]:
contacts.drop_duplicates(subset=contacts.columns[1:], inplace=True)

###Results control

In [ ]:
contacts.isna().sum()

In [ ]:
contacts.info()

NameError: name 'contacts' is not defined

## **Calls**

In [ ]:
calls.head()

NameError: name 'calls' is not defined

###Datatype changes

In [ ]:
calls['Call Start Time'] = pd.to_datetime(calls['Call Start Time'], errors='raise')

###Drop duplicates

In [ ]:
calls.drop_duplicates(subset=calls.columns[1:], inplace=True)
duplicate_mask_columns = [
    'Call Start Time', 'Call Owner Name', 'Call Duration (in seconds)',
    'Call Status'
]
calls.drop_duplicates(subset=duplicate_mask_columns, keep='first', inplace=True)

NameError: name 'calls' is not defined

###Data transformation

In [ ]:
calls['Call Duration (in min)'] = (calls['Call Duration (in seconds)'] / 60).round(2)

NameError: name 'calls' is not defined

###Drop columns

In [ ]:
calls = calls.drop(['Dialled Number', 'Tag', 'Call Duration (in seconds)'], axis=1)

###Fill NaN values

In [ ]:
categories = [
    'Call Owner Name', 'Call Type', 'Call Status', 'Outgoing Call Status',
    'Scheduled in CRM'
]
for col in categories:
    calls[col] = calls[col].fillna('Unknown')

###Results control

In [ ]:
calls.isna().sum()

In [ ]:
calls.info()

NameError: name 'calls' is not defined

## **Spend**

In [ ]:
spend.head()

NameError: name 'spend' is not defined

###Datatype changes

In [ ]:
spend["Date"] = pd.to_datetime(spend["Date"], errors="raise")

###Drop duplicates

In [ ]:
spend.drop_duplicates(subset=spend.columns[1:], inplace=True)

###Data transformation

In [ ]:
spend['Spend'] = spend['Spend'].replace(r'[€]', '', regex=True).astype(float)

In [ ]:
spend = spend[spend['Source'] != 'Test']

In [ ]:
spend = spend[
    (spend['Impressions'] != 0) &
    (spend['Spend'] != 0) &
    (spend['Clicks'] != 0)
]

NameError: name 'spend' is not defined

###Fill NaN values

In [ ]:
categories = ['Campaign', 'AdGroup', 'Ad']
for col in categories:
    spend[col] = spend[col].fillna('Unknown')

NameError: name 'spend' is not defined

###Results control

In [ ]:
spend.info()

NameError: name 'spend' is not defined

In [ ]:
spend.isna().sum()

## **Deals**

In [ ]:
deals.head()

###Datatype changes

In [ ]:
deals['Closing Date'] = pd.to_datetime(deals['Closing Date'], format='%d.%m.%Y',
                                       errors='raise')
deals['Created Time'] = pd.to_datetime(deals['Created Time'],
                                       format='%d.%m.%Y %H:%M',
                                       errors='raise')

NameError: name 'deals' is not defined

###Drop duplicates

In [ ]:
deals.drop_duplicates(subset=deals.columns[1:], inplace=True)

NameError: name 'deals' is not defined

###Data transformation

In [ ]:
mask = deals['Created Time'].dt.date > deals['Closing Date']
mask.sum()

In [ ]:
deals.loc[mask, 'Created Time'] = deals.loc[mask, 'Closing Date']
deals.loc[mask, 'Closing Date'] = deals.loc[mask, 'Created Time'].dt.date

In [ ]:
mask.sum()

In [ ]:
def convert_to_minutes(x):
    """
    The function processes various time formats and converts them into a total
    number of minutes.
    """
    if pd.isna(x):
        return np.nan
    elif isinstance(x, datetime.time):
        return x.hour * 60 + x.minute + x.second / 60
    elif isinstance(x, datetime.timedelta):
        return x.total_seconds() / 60

deals['SLA Minutes'] = deals['SLA'].apply(convert_to_minutes).round(2)

NameError: name 'deals' is not defined

In [ ]:
def clean_currency_columns(df, columns_to_clean):
    """
    The function removes non-numeric characters from columns containing financial
    figures and converts the values to the float data type
    """
    for col in columns_to_clean:
        df[col] = (df[col]
            .replace(r'[€]', '', regex=True)
            .replace(r'\s+', '', regex=True)
            .replace(r'\.', '', regex=True)
            .replace(r',', '.', regex=True)
            .astype(float)
        )
    return df

columns_to_clean = ['Initial Amount Paid', 'Offer Total Amount']
deals = clean_currency_columns(deals, columns_to_clean)

In [ ]:
mask = deals['Initial Amount Paid'] > deals['Offer Total Amount']
deals.loc[mask, 'Initial Amount Paid'] = deals.loc[mask, 'Offer Total Amount']
deals.loc[mask, 'Offer Total Amount'] = deals.loc[mask, 'Initial Amount Paid']

NameError: name 'deals' is not defined

###Unnecessary values removing

In [ ]:
deals.drop(columns=['SLA'], inplace=True)

NameError: name 'deals' is not defined

In [ ]:
deals = deals.replace('#REF!', pd.NA)
deals = deals.dropna(how='all')

NameError: name 'deals' is not defined

In [ ]:
deals = deals[deals['Created Time'] >= '2023-01-01']

NameError: name 'deals' is not defined

In [ ]:
deals['Product'].value_counts()

NameError: name 'deals' is not defined

In [ ]:
deals = deals[~(deals['Product'].isin(['Find yourself in IT', 'Data Analytics']))]

NameError: name 'deals' is not defined

In [ ]:
deals['Page'].value_counts()

NameError: name 'deals' is not defined

In [ ]:
#deals = deals[~(deals['Page'].isin(['/eng/test', '/test']))]

In [ ]:
deals['Source'].value_counts()

NameError: name 'deals' is not defined

In [ ]:
deals = deals[deals['Source'] != 'Test']

NameError: name 'deals' is not defined

In [ ]:
deals['Lost Reason'].value_counts()

NameError: name 'deals' is not defined

In [ ]:
deals = deals[deals['Lost Reason'] != 'Duplicate']

###City processing

In [ ]:
mode_values = deals.groupby('Contact Name')['City'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)
deals['City'] = deals['Contact Name'].map(mode_values)

NameError: name 'deals' is not defined

In [ ]:
deals['City'].unique().tolist()[:10]

NameError: name 'deals' is not defined

In [ ]:
deals['City'] = deals['City'].replace('-', 'Unknown')

NameError: name 'deals' is not defined

In [ ]:
with open('city_data_google_en.json', 'r') as json_file:
    city_data = json.load(json_file)

FileNotFoundError: [Errno 2] No such file or directory: 'city_data_google_en.json'

In [ ]:
def get_city_info(city):
    """
    The function searches for a city in the global `city_data` dictionary and
    returns key geographical and administrative data as a pandas series.
    """
    info = city_data.get(city, {})
    return pd.Series({
        'longitude': info.get('longitude', None),
        'latitude': info.get('latitude', None),
        'country': info.get('country', None),
        'federal_state': info.get('federal_state', None),
        'city_en': info.get('city', None)
})

deals[['longitude', 'latitude', 'city_en', 'country', 'federal_state']] = deals['City'].apply(get_city_info)

NameError: name 'deals' is not defined

###Level of Deutsch processing

In [ ]:
mode_values = deals.groupby('Contact Name')['Level of Deutsch'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)
deals['Level of Deutsch'] = deals['Contact Name'].map(mode_values)

NameError: name 'deals' is not defined

In [ ]:
deals['Level of Deutsch'].unique().tolist()[:10]

NameError: name 'deals' is not defined

In [ ]:
with open('deutsch_level_dict.json', 'r', encoding='utf-8') as f:
    level_mapping = json.load(f)

deals['Level of Deutsch'] = deals[
    'Level of Deutsch'].map(level_mapping).fillna('Unknown')

FileNotFoundError: [Errno 2] No such file or directory: 'deutsch_level_dict.json'

In [ ]:
deals_category = deals.select_dtypes(include=['object']).columns[1:]
deals[deals_category] = deals[deals_category].fillna('Unknown')

In [ ]:
deals.info()

In [ ]:
deals.isna().sum()

NameError: name 'deals' is not defined

# **Descriptive statistics**

##Calls statistic

In [ ]:
calls_desc = calls[['Call Duration (in min)']].describe().T
calls_desc['mode'] = calls['Call Duration (in min)'].mode().iloc[0]
calls_desc

NameError: name 'calls' is not defined

In [ ]:
call_status = pd.DataFrame({
    'Count': calls['Call Status'].value_counts(),
    'Percentage': calls['Call Status'].value_counts(normalize=True) * 100
}).round(2).T

call_status

NameError: name 'calls' is not defined

Die meisten Anrufe sind sehr kurz, meistens dauern sie 0 Minuten. Anhand der Prozentsätze der angenommenen und nicht angenommenen Anrufe lässt sich erkennen, dass die meisten Anrufe angenommen, aber schnell beendet werden. Eine weitere Analyse mit einer Gruppierung der Anrufe nach Kontakten ist erforderlich, um zu verstehen, ob nach den kurzen Anrufen längere Gespräche mit diesen Kontakten stattfanden.

##Spend statistic

In [ ]:
spend_numeric = ['Impressions', 'Spend', 'Clicks']
spend_desc = spend[spend_numeric].describe().T
spend_desc['mode'] = spend[spend_numeric].mode().iloc[0].values
spend_desc

NameError: name 'spend' is not defined

Der große Unterschied zwischen dem Medianwert und dem Durchschnittswert aller analysierten Indikatoren deutet auf große Schwankungen hin. Eine weitere Analyse mit einer Gruppierung nach Unternehmen und Werbequellen ist erforderlich.

##Deals statistic

In [ ]:
deals_numeric = deals.select_dtypes(include=['number']).columns
deals_desc = deals[deals_numeric].describe().T
deals_desc['mode'] = deals[deals_numeric].mode().iloc[0].values
deals_desc

NameError: name 'deals' is not defined

Es ist ersichtlich, dass alle Indikatoren mit Ausnahme der Reaktionszeit relativ standardisiert sind. Längen- und Breitengrade weisen eine geringe Streuung auf, die sich auf Europa konzentriert. Der häufigste Wert entspricht Berlin.

Die SLA-Reaktionszeit weist eine enorme Streuung der Werte auf. Ersetzen wir 0,05 % der Maximalwerte durch die nächste Obergrenze

In [ ]:
upper_bound = deals['SLA Minutes'].quantile(0.995)
deals.loc[deals['SLA Minutes'] > upper_bound, 'SLA Minutes'] = upper_bound

NameError: name 'deals' is not defined

In [ ]:
deals_numeric = deals.select_dtypes(include=['number']).columns
deals_desc = deals[deals_numeric].describe().T
deals_desc['mode'] = deals[deals_numeric].mode().iloc[0].values
deals_desc

NameError: name 'deals' is not defined

Die Verarbeitung von 0,05 % der SLA-Minuten-Ausfälle hat die Verteilung verbessert. Für die weitere Analyse werden die verarbeiteten Werte verwendet.

In [ ]:
cat_columns = ['Stage', 'Source', 'Quality', 'Product']
for col in cat_columns:
    deals_cat = pd.DataFrame({
    'Count': deals[col].value_counts(),
    'Percentage': deals[col].value_counts(normalize=True) * 100
}).round(2).T
    print(col)
    display(deals_cat)

NameError: name 'deals' is not defined

Der Prozentsatz der verlorenen Transaktionen („Lost“, „Call Delayed“) ist mit 82 % sehr hoch. Die Konversionsrate in die Phase des erfolgreichen Abschlusses („Payment Done“) beträgt 4 %.

Dabei scheint die Anzahl der Transaktionen mit hoher und niedriger Qualität in quantitativer und prozentualer Hinsicht in dieser Phase keinen Zusammenhang mit der tatsächlichen Anzahl der abgeschlossenen und verlorenen Transaktionen zu haben.

Es ist eine zusätzliche Analyse der Manager erforderlich, um die Bedeutung des Parameters „Quality” zu bewerten.

Der Wert „Product” ist in 82 % der Fälle unbekannt, was möglicherweise mit der Anzahl der verlorenen Transaktionen („Lost”, „Call Delayed”) und derjenigen zusammenhängt, die sich in der Phase der Kundeninformation über die Produkte befinden („Registered on Webinar”). Von den bekannten Produkten ist „Digital Marketing” am beliebtesten. Eine zusätzliche Analyse der übrigen Produkte ist erforderlich.

Die wichtigsten Kanäle zur Kundengewinnung sind Facebook Ads (24 %) und Google Ads (21 %). Die Aktivität der übrigen Quellen ist um das Zweifache oder mehr geringer. Es ist eine Bewertung der Effektivität der Kanäle hinsichtlich der Kosten für die Gewinnung zahlungskräftiger Kunden und der Konversion in Zahlungen erforderlich.

# **Cleaned data saving**

In [ ]:
calls.to_parquet('calls_clean.parquet')
spend.to_parquet('spend_clean.parquet')
deals.to_parquet('deals_clean.parquet')
contacts.to_parquet('contacts_clean.parquet')